# Gray-Scott Model: Reaction-Diffusion System

---
## **Introduction to the Gray-Scott Model**

The **Gray-Scott model** is a **reaction-diffusion system** that describes the interaction of two chemical species, **U** and **V**, whose concentrations change over time and space due to **reaction** and **diffusion**.

### **Governing Equations**

The system is described by the following partial differential equations (PDEs):

$$
\begin{cases}
\displaystyle \frac{\partial u}{\partial t} = D_u \Delta u - uv^2 + F(1 - u) \\
\displaystyle \frac{\partial v}{\partial t} = D_v \Delta v + uv^2 - (F + k)v
\end{cases}
$$

**Terms:**
- **$D_u$ and $D_v$**: Diffusion coefficients for **U** and **V**, respectively.
- **$F$**: Feed rate (inflow of **U**).
- **$k$**: Kill rate (outflow of **V**).
- **$\Delta$**: Laplacian operator, representing diffusion.
- **$uv^2$**: Reaction term where **U** and **V** interact.

---
## **Numerical Scheme**

### **Laplacian Discretization**
The Laplacian is approximated using a **second-order finite difference scheme**:

$$
\Delta u_{i,j} \approx u_{i,j-1} + u_{i-1,j} - 4u_{i,j} + u_{i+1,j} + u_{i,j+1}
$$

### **Time Integration**
The **Euler scheme** is used for time integration to update the concentrations of **U** and **V** at each time step.

---
## **Initialization**

The initial conditions are:
- **$u = 1$** everywhere.
- **$v = 0$** everywhere, except in a **central square** of size $0.2 \times 0.2$ where:
  - **$u = 0.5$**
  - **$v = 0.25$**

**Implementation:**

In [ ]:
import numpy as np
%config InlineBackend.figure_format = 'retina'

def init(n):
    """Initialize the concentration fields u and v."""
    u = np.ones((n + 2, n + 2))  # u = 1 everywhere
    v = np.zeros((n + 2, n + 2))  # v = 0 everywhere

    # Create a grid
    x, y = np.meshgrid(np.linspace(0, 1, n + 2), np.linspace(0, 1, n + 2))

    # Define the central square
    mask = (0.4 < x) & (x < 0.6) & (0.4 < y) & (y < 0.6)

    # Set initial values in the central square
    u[mask] = 0.50
    v[mask] = 0.25

    return u, v

---
## **Boundary Conditions**

The domain is assumed to be **periodic**, meaning the values at the boundaries wrap around to the opposite side.

**Implementation:**

In [ ]:
def periodic_bc(u):
    """Apply periodic boundary conditions to the field u."""
    u[0, :] = u[-2, :]  # Top boundary
    u[-1, :] = u[1, :]  # Bottom boundary
    u[:, 0] = u[:, -2]  # Left boundary
    u[:, -1] = u[:, 1]  # Right boundary

---
## **Laplacian Calculation**

The Laplacian is computed using a **second-order finite difference scheme**.

**Implementation:**

In [ ]:
def laplacian(u):
    """Compute the Laplacian of u using finite differences."""
    return (
        u[:-2, 1:-1] +
        u[1:-1, :-2] -
        4 * u[1:-1, 1:-1] +
        u[1:-1, 2:] +
        u[2:, 1:-1]
    )

---
## **Gray-Scott Model Implementation**

The `grayscott` function updates the concentrations of **U** and **V** based on the reaction-diffusion equations.

**Implementation:**

In [ ]:
def grayscott(U, V, Du, Dv, F, k):
    """Update the concentrations of U and V using the Gray-Scott model."""
    u, v = U[1:-1, 1:-1], V[1:-1, 1:-1]  # Extract inner fields

    # Compute Laplacians
    Lu = laplacian(U)
    Lv = laplacian(V)

    # Reaction terms
    uvv = u * v * v

    # Update U and V
    U[1:-1, 1:-1] += Du * Lu - uvv + F * (1 - u)
    V[1:-1, 1:-1] += Dv * Lv + uvv - (F + k) * v

    # Apply boundary conditions
    periodic_bc(U)
    periodic_bc(V)

---
## **Simulation Parameters**

The following parameters are used for the simulation:
- **$D_u = 0.1$**: Diffusion coefficient for **U**.
- **$D_v = 0.05$**: Diffusion coefficient for **V**.
- **$F = 0.0545$**: Feed rate.
- **$k = 0.062$**: Kill rate.

In [ ]:
Du, Dv = 0.1, 0.05
F, k = 0.0545, 0.062

---
## **Running the Simulation**

The simulation is run for a specified number of time steps, and the results are visualized as an animation.

**Implementation:**

In [ ]:
from tqdm.notebook import tqdm
from PIL import Image

# Initialize fields
U, V = init(300)

def create_image():
    """Create an image from the current state of V."""
    global U, V
    for _ in range(40):  # Update 40 time steps per frame
        grayscott(U, V, Du, Dv, F, k)

    # Scale V to 0-255 for visualization
    V_scaled = np.uint8(255 * (V - V.min()) / (V.max() - V.min()))
    return V_scaled

def create_frames(n):
    """Create a sequence of frames for the animation."""
    return [create_image() for _ in tqdm(range(n))]

# Generate frames
frames = create_frames(500)

---
## **Visualization**

### **Interactive Visualization**
Use an interactive slider to explore the simulation frames.

**Implementation:**

In [ ]:
from ipywidgets import interact, IntSlider

def display_sequence(iframe):
    """Display a specific frame from the simulation."""
    return Image.fromarray(frames[iframe])

interact(
    display_sequence,
    iframe=IntSlider(
        min=0,
        max=len(frames) - 1,
        step=1,
        value=0,
        continuous_update=True
    )
)

---
### **Saving the Animation**

The frames can be saved as a **GIF** for easy sharing and visualization.

**Implementation:**

In [ ]:
import imageio

# Scale frames to 0-255
frames_scaled = [np.uint8(255 * frame) for frame in frames]

# Save as GIF
imageio.mimsave('images/movie.gif', frames_scaled, format='gif', fps=60)

**Display the GIF:**

In [ ]:
from IPython.display import HTML
HTML('<img src="images/movie.gif">')

**Output:** An animated GIF of the Gray-Scott model simulation.

---
## **References**

For further reading, check out:
- **[Reaction-Diffusion by the Gray-Scott Model: Pearson's Parametrization](https://mrob.com/pub/comp/xmorphia/)**: A detailed explanation of the Gray-Scott model and its patterns.

---
```